# 🚀 Running RAGStack on Google Colab with GPU Acceleration

This notebook guide helps you run the **RAGStack Benchmarking Pipeline** on a cloud GPU instance in Google Colab.

### ⚠️ Setup GPU Runtime First
Before running any cells, ensure you are connected to a GPU instance:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Under *Hardware accelerator*, select **T4 GPU** (or any other available GPU).
3. Click **Save**.

## 📂 Step 1: Set Up Project Workspace

Choose **Option A** if you cloned the repository to your local Google Drive, or **Option B** if you want to clone it from GitHub.

In [ ]:
# === OPTION A: Run from Google Drive ===
# Mount your Drive and navigate to the project directory
from google.colab import drive
drive.mount('/content/drive')

# Update this path to point to your ragstack directory in Drive
%cd /content/drive/MyDrive/ragstack

In [ ]:
# === OPTION B: Clone from GitHub ===
# Uncomment and run this if you want to pull a fresh copy from GitHub
# !git clone <YOUR_GITHUB_REPO_URL> ragstack
# %cd ragstack

## 📦 Step 2: Install Dependencies

Colab environments come with PyTorch and Pandas pre-installed, but we need to install the project-specific dependencies (`langchain`, `chromadb`, `sentence-transformers`, etc.).

In [ ]:
!pip install -r requirements.txt

## ⚙️ Step 3: Run the Benchmarking Pipeline

We will run the pipeline using the `--profile colab-gpu` flag. This configures the pipeline to use GPU-accelerated local models:
- **Embedder**: `bge-small` (`BAAI/bge-small-en-v1.5` via SentenceTransformers)
- **Generator**: `hf-small` (`google/flan-t5-base` via Hugging Face pipeline)
- **Evaluator**: `llm` (LLM-as-a-judge scoring via `google/flan-t5-base`)

### 1. Run a single evaluation test run
Let's run a test on the `covidqa` dataset with `n = 10` records to verify the pipeline runs end-to-end:

### 0. Define Run Session ID
Define a unique `RUN_ID_PREFIX` to group all runs in this session together (e.g., for plotting and reports):

In [ ]:
import time
RUN_ID_PREFIX = f"{time.strftime('%Y%m%d_%H%M')}_colab_user_covidqa"
print(f"RUN_ID_PREFIX set to: {RUN_ID_PREFIX}")

In [ ]:
!python main.py --profile colab-gpu --dataset covidqa -n 10 --username colab_user --run_id_prefix {RUN_ID_PREFIX}

### 2. Run a full configuration sweep
Use the `--sweep` flag to benchmark different combinations of **chunkers** (`recursive`, `semantic`) and **retrievers** (`dense`, `hybrid`) on the dataset:

In [ ]:
!python main.py --profile colab-gpu --dataset covidqa -n 10 --sweep --username colab_user --run_id_prefix {RUN_ID_PREFIX}

## 📊 Step 4: Analyze & Compare Results

Let's load the updated leaderboard results and visualize the performance of your latest run.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load master tracking file
master_path = "master_tracking.csv"
if os.path.exists(master_path):
    df = pd.read_csv(master_path)
    colab_runs = df[df['embedder'].isin(['bge-small', 'bge-large'])]
    
    print(f"Total benchmark runs logged: {len(df)}")
    print(f"Colab GPU benchmark runs: {len(colab_runs)}")
    
    # Sort and display leaderboard by mean adherence & context relevance
    leaderboard = df.sort_values(by=["mean_adherence", "mean_context_relevance"], ascending=False)
    display(leaderboard)
else:
    print(f"No leaderboard found at '{master_path}'. Run the pipeline cells above first.")

### Plot performance of the current run

This cell extracts the run IDs executed in the latest run from the `eval/latest_run.txt` marker file and renders the corresponding plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists(master_path) and len(df) > 0:
    # Parse timestamp
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.sort_values(by='timestamp_dt', ascending=True)
    
    latest_run_file = "eval/latest_run.txt"
    current_run_df = pd.DataFrame()
    latest_sweep_id = None
    latest_dataset = None
    
    def get_sweep_prefix(run_id, dataset):
        if not isinstance(dataset, str) or not isinstance(run_id, str):
            return run_id
        idx = run_id.find(dataset)
        if idx != -1:
            return run_id[:idx + len(dataset)]
        return run_id

    # Option 1: Retrieve from RUN_ID_PREFIX python variable defined in the session
    if 'RUN_ID_PREFIX' in globals() and RUN_ID_PREFIX:
        current_run_df = df[df['run_id'].str.startswith(RUN_ID_PREFIX, na=False)]
        if not current_run_df.empty:
            latest_sweep_id = RUN_ID_PREFIX
            latest_dataset = current_run_df.iloc[0]['dataset']
            print(f"Loaded {len(current_run_df)} runs matching RUN_ID_PREFIX '{RUN_ID_PREFIX}'")

    # Option 2: Retrieve exact run IDs from latest_run.txt marker file
    if os.path.exists(latest_run_file):
        try:
            with open(latest_run_file, "r") as f:
                latest_run_ids = [line.strip() for line in f.read().splitlines() if line.strip()]

            current_run_df = df[df['run_id'].isin(latest_run_ids)]
            if not current_run_df.empty:
                first_row = current_run_df.iloc[0]
                latest_dataset = first_row['dataset']
                earliest_run = current_run_df.sort_values(by='timestamp_dt').iloc[0]
                latest_sweep_id = get_sweep_prefix(earliest_run['run_id'], latest_dataset)
                print(f"Loaded {len(current_run_df)} runs from marker file '{latest_run_file}'")
        except Exception as e:
            print(f"Warning: Failed to read latest_run.txt ({e}). Falling back to time grouping.")
            current_run_df = pd.DataFrame()

    # Option 3: Fallback to grouping latest runs within a 15-minute time window
    if current_run_df.empty and not df.empty:
        latest_run = df.iloc[-1]
        latest_time = latest_run['timestamp_dt']
        latest_dataset = latest_run['dataset']
        latest_user = latest_run['username']
        
        time_threshold = pd.Timedelta(minutes=15)
        current_run_df = df[
            (df['dataset'] == latest_dataset) & 
            (df['username'] == latest_user) & 
            ((latest_time - df['timestamp_dt']) <= time_threshold) & 
            ((latest_time - df['timestamp_dt']) >= pd.Timedelta(seconds=0))
        ]
        earliest_run = current_run_df.sort_values(by='timestamp_dt').iloc[0]
        latest_sweep_id = get_sweep_prefix(earliest_run['run_id'], latest_dataset)
        print(f"Grouped {len(current_run_df)} runs from latest 15-minute time window.")

    if not current_run_df.empty:
        print(f"Plotting Sweep ID: {latest_sweep_id} | Dataset: {latest_dataset}")
        
        # Exclude mock runs unless they are the only ones
        colab_runs = current_run_df[current_run_df['embedder'].isin(['bge-small', 'bge-large'])]
        if colab_runs.empty:
            colab_runs = current_run_df
            
        # Set plotting theme
        sns.set_theme(style="whitegrid")
        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
        
        # 1. Adherence by Configuration
        sns.barplot(ax=axes[0, 0], data=colab_runs, x="chunker", y="mean_adherence", hue="retriever", palette="muted")
        axes[0, 0].set_title("Mean Adherence")
        axes[0, 0].set_ylim(0, 1.0)
        
        # 2. Context Relevance
        sns.barplot(ax=axes[0, 1], data=colab_runs, x="chunker", y="mean_context_relevance", hue="retriever", palette="muted")
        axes[0, 1].set_title("Context Relevance")
        axes[0, 1].set_ylim(0, 1.0)

        # 3. Context Utilization
        sns.barplot(ax=axes[1, 0], data=colab_runs, x="chunker", y="mean_context_utilization", hue="retriever", palette="muted")
        axes[1, 0].set_title("Context Utilization")
        axes[1, 0].set_ylim(0, 1.0)
        
        # 4. Latency
        sns.barplot(ax=axes[1, 1], data=colab_runs, x="chunker", y="mean_latency_ms", hue="retriever", palette="muted")
        axes[1, 1].set_title("Mean Latency (ms)")
        
        plt.suptitle(f"RAG Benchmark Performance\nSweep ID: {latest_sweep_id}", fontsize=14, fontweight='bold')
        
        # Format overlay textbox
        run_details = []
        for _, row in colab_runs.sort_values(by="mean_adherence", ascending=False).iterrows():
            run_details.append(
                f"• {row['chunker']}/{row['retriever']} ({row['embedder']}) -> "
                f"Adh: {row['mean_adherence']:.2f}, Lat: {row['mean_latency_ms']:.0f}ms"
            )
        run_details_str = "Run Configurations Plotted:\n" + "\n".join(run_details)
        
        fig.text(
            0.05, 0.01, run_details_str, 
            fontsize=8, family='monospace', 
            bbox=dict(facecolor='white', alpha=0.9, boxstyle='round,pad=0.5', edgecolor='gray')
        )
        
        plt.subplots_adjust(bottom=0.16)
        
        # Save plot image
        os.makedirs("eval", exist_ok=True)
        plot_path = f"eval/benchmark_plot_{latest_sweep_id}.png"
        plt.savefig(plot_path, dpi=300)
        print(f"Plot successfully saved to '{plot_path}'")
        plt.show()
    else:
        print("No runs found.")
else:
    print("Run the pipeline first to generate metrics.")

## 📥 Step 5: Export & Download Results

Run the cell below to package the leaderboard (`master_tracking.csv`), detailed run files (`eval/`), and the benchmark plot image into a zip file, and trigger a download to your local machine.

You can then run the local script `python scripts/merge_results.py` to merge these results directly into your local Git repository.

In [ ]:
# Pack results into a zip
!zip -q -r eval_results.zip master_tracking.csv eval/

# Trigger browser download
from google.colab import files
try:
    files.download('eval_results.zip')
    print("Triggered download of eval_results.zip successfully.")
except Exception as e:
    print(f"Could not automatically trigger download: {e}")
    print("You can manually download 'eval_results.zip' from the left file explorer sidebar.")